# Reproducing the results

Recomputes every reported quantity from the stored detections and the field inventory, and
regenerates both figures. Sections follow the order of the analysis. Steps requiring a GPU are shown as
the command that produced the stored artifact.

In [1]:
import itertools, json, os, sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
import matplotlib.pyplot as plt

HERE = os.path.abspath("..")
DATA = os.environ.get("TREELEARN_DATA", HERE)
ROOT = os.environ.get("TREELEARN_ROOT", HERE)
RESULTS = os.path.join(HERE, "results")
sys.path.insert(0, os.path.join(HERE, "modal"))
sys.path.insert(0, os.path.join(HERE, "analysis"))

MET = ("SAT", "AMS3D", "FF3D", "TreeLite3D", "WS")
NAME = {"SAT": "SegmentAnyTree", "AMS3D": "AMS3D", "FF3D": "ForestFormer3D",
        "TreeLite3D": "TreeLite3D", "WS": "CHM watershed"}
COL = {"SAT": "#1b3b6f", "AMS3D": "#d1701a", "FF3D": "#2e7d5b",
       "TreeLite3D": "#8e44ad", "WS": "#7a8b7f"}
MRK = {"SAT": "o", "AMS3D": "s", "FF3D": "D", "TreeLite3D": "v", "WS": "^"}
DMATCH, NBOOT, BUF = 3.5, 2000, 1.5

DET_FILE = os.path.join(ROOT, "all_detections.json")
FULL = os.path.exists(DET_FILE)
if FULL:
    from evaluation_core import reference, region
    A = json.load(open(DET_FILE))
    DET, DENS = A["detections"], A["density"]
    KEYS = list(DET)
    REF = {k: reference(*k.split("/")) for k in KEYS}
    REG = {k: region(REF[k], BUF) for k in KEYS}
else:
    KEYS = []
MAIN = json.load(open(os.path.join(RESULTS, "main_evaluation.json")))
SKIP = ("skipped: this step recomputes from all_detections.json, which is not here.\n"
        "        Set TREELEARN_ROOT, or follow the README to rebuild it.")
print("full recompute" if FULL else "published results only", "|", len(KEYS), "plots")
if not FULL:
    print(SKIP.splitlines()[0])

full recompute | 22 plots


## 1. Data and reference

In [2]:
if not FULL:
    print(SKIP)
else:
    GRP = {"Chamrousse": [k for k in KEYS if k.startswith("cham")],
           "Pellizzano": [k for k in KEYS if k.startswith("pell")],
           "Both": KEYS}
    for g, ks in GRP.items():
        d = [DENS[k]["density"] for k in ks]
        n = sum(len(REF[k]["x"]) for k in ks)
        print(f"{g:11} {len(ks):>3} plots {n:>5} trees   density {min(d):.1f}-{max(d):.1f} returns/m2")

    d = np.array([DENS[k]["density"] for k in KEYS])
    t = np.array([len(REF[k]["x"]) for k in KEYS])
    a = np.array([DENS[k]["area"] for k in KEYS])

Chamrousse    7 plots   894 trees   density 36.1-70.6 returns/m2
Pellizzano   15 plots   543 trees   density 90.8-172.9 returns/m2
Both         22 plots  1437 trees   density 36.1-172.9 returns/m2


In [3]:
if not FULL:
    print(SKIP)
else:
    def spearman(x, y, nperm=10000, seed=0):
        x, y = np.asarray(x, float), np.asarray(y, float)
        rx, ry = x.argsort().argsort() + 1.0, y.argsort().argsort() + 1.0
        rho = float(np.corrcoef(rx, ry)[0, 1])
        rng = np.random.default_rng(seed)
        hits = sum(abs(np.corrcoef(rng.permutation(rx), ry)[0, 1]) >= abs(rho)
                   for _ in range(nperm))
        return rho, (hits + 1) / (nperm + 1)


    for lab, v in (("tree count", t), ("region area", a)):
        r, p = spearman(d, v)
        print(f"density vs {lab:12} rho {r:+.3f}  p {p:.4f}")
    print(f"region / tile: {np.mean([DENS[k]['area'] / DENS[k].get('tile_area', DENS[k]['area']) for k in KEYS]):.3f}"
          if "tile_area" in DENS[KEYS[0]] else "")

density vs tree count   rho -0.730  p 0.0002


density vs region area  rho -0.609  p 0.0020



## 2. Matching protocol

In [4]:
if not FULL:
    print(SKIP)
else:
    def greedy(dx, dy, rx, ry, dmax=DMATCH):
        if not len(dx) or not len(rx):
            return []
        D = np.hypot(np.asarray(dx)[:, None] - rx[None, :], np.asarray(dy)[:, None] - ry[None, :])
        cand = np.argwhere(D <= dmax)
        order = np.argsort(D[cand[:, 0], cand[:, 1]], kind="stable")
        ud, ur, out = set(), set(), []
        for i, j in cand[order]:
            if i in ud or j in ur:
                continue
            ud.add(i); ur.add(j); out.append((int(i), int(j)))
        return out


    def metrics(TP, NR, ND):
        r = TP / NR if NR else 0.0
        p = TP / ND if ND else 0.0
        return r, p, (2 * p * r / (p + r) if p + r else 0.0)


    def per_plot(k, met, dmax=DMATCH):
        d = DET[k][met]
        x, y = np.array(d["x"]), np.array(d["y"])
        ins = REG[k].contains_points(np.c_[x, y]) if len(x) else np.zeros(0, bool)
        pairs = greedy(x[ins], y[ins], REF[k]["x"], REF[k]["y"], dmax)
        return dict(TP=len(pairs), ND=int(ins.sum()), NR=len(REF[k]["x"]), pairs=pairs)


    TAB = {m: {k: per_plot(k, m) for k in KEYS} for m in MET}

## 3. Main comparison

In [5]:
if not FULL:
    print(SKIP)
else:
    def pooled(met, keys, tab=None):
        v = [(tab or TAB)[met][k] for k in keys]
        TP, ND, NR = (sum(x[c] for x in v) for c in ("TP", "ND", "NR"))
        return TP, ND, NR, *metrics(TP, NR, ND)


    def boot_recall(met, keys, nb=NBOOT, seed=0):
        v = [TAB[met][k] for k in keys]
        rng = np.random.default_rng(seed)
        out = np.empty(nb)
        for b in range(nb):
            i = rng.integers(0, len(v), len(v))
            num = sum(v[j]["TP"] for j in i); den = sum(v[j]["NR"] for j in i)
            out[b] = num / den if den else 0.0
        return np.percentile(out, [2.5, 97.5])


    for g, keys in GRP.items():
        print(f"\n{g}")
        print(f"{'method':16} {'TP':>5} {'recall':>7} {'95% CI':>16} {'prec.':>7} {'F1':>7}")
        for m in MET:
            TP, ND, NR, r, p, f = pooled(m, keys)
            lo, hi = boot_recall(m, keys)
            print(f"{NAME[m]:16} {TP:>5} {r:>7.3f} [{lo:.3f}, {hi:.3f}] {p:>7.3f} {f:>7.3f}")


Chamrousse
method              TP  recall           95% CI   prec.      F1
SegmentAnyTree     632   0.707 [0.630, 0.765]   0.817   0.758
AMS3D              575   0.643 [0.569, 0.709]   0.833   0.726
ForestFormer3D     536   0.600 [0.479, 0.657]   0.841   0.700
TreeLite3D         442   0.494 [0.424, 0.598]   0.745   0.594
CHM watershed      386   0.432 [0.383, 0.496]   0.910   0.586

Pellizzano
method              TP  recall           95% CI   prec.      F1
SegmentAnyTree     339   0.624 [0.542, 0.734]   0.887   0.733
AMS3D              349   0.643 [0.567, 0.740]   0.831   0.725
ForestFormer3D     342   0.630 [0.558, 0.739]   0.838   0.719
TreeLite3D         368   0.678 [0.607, 0.746]   0.639   0.658
CHM watershed      203   0.374 [0.317, 0.456]   0.990   0.543

Both
method              TP  recall           95% CI   prec.      F1


SegmentAnyTree     971   0.676 [0.605, 0.727]   0.840   0.749
AMS3D              924   0.643 [0.590, 0.694]   0.832   0.726
ForestFormer3D     878   0.611 [0.539, 0.664]   0.840   0.707
TreeLite3D         810   0.564 [0.506, 0.667]   0.693   0.622
CHM watershed      589   0.410 [0.366, 0.452]   0.936   0.570


## 4. Chance floor

In [6]:
if not FULL:
    print(SKIP)
else:
    def uniform_recall(n, reg, ref, dmatch, rng, reps=20):
        if not n:
            return 0.0
        v = reg.vertices
        x0, x1, y0, y1 = v[:, 0].min(), v[:, 0].max(), v[:, 1].min(), v[:, 1].max()
        tot = 0
        for _ in range(reps):
            got = []
            while len(got) < n:
                c = np.c_[rng.uniform(x0, x1, n * 3), rng.uniform(y0, y1, n * 3)]
                got.extend(c[reg.contains_points(c)].tolist())
            c = np.array(got[:n])
            tot += len(greedy(c[:, 0], c[:, 1], ref["x"], ref["y"], dmatch))
        return tot / reps


    for site in ("cham", "pell"):
        s = [k for k in KEYS if k.startswith(site)]
        nn = []
        for k in s:
            x, y = np.asarray(REF[k]["x"]), np.asarray(REF[k]["y"])
            D = np.hypot(x[:, None] - x[None, :], y[:, None] - y[None, :])
            np.fill_diagonal(D, np.inf)
            nn.extend(D.min(axis=1))
        nn = np.array(nn)
        print(f"{site}: nearest stem median {np.median(nn):.2f} m, "
              f"{(nn <= DMATCH).mean():.0%} within {DMATCH} m")

cham: nearest stem median 1.89 m, 82% within 3.5 m
pell: nearest stem median 1.98 m, 81% within 3.5 m


In [7]:
if not FULL:
    print(SKIP)
else:
    rng = np.random.default_rng(0)
    print(f"{'method':16} {'recall':>7} {'chance':>7} {'excess':>7} {'prec.':>7}")
    FLOOR = {}
    for m in MET:
        tp = tp0 = nd = nr = 0
        for k in KEYS:
            v = TAB[m][k]
            tp += v["TP"]; nd += v["ND"]; nr += v["NR"]
            tp0 += uniform_recall(v["ND"], REG[k], REF[k], DMATCH, rng)
        FLOOR[m] = dict(recall=tp / nr, chance=tp0 / nr, excess=tp / nr - tp0 / nr)
        print(f"{NAME[m]:16} {tp/nr:>7.3f} {tp0/nr:>7.3f} {tp/nr - tp0/nr:>7.3f} {tp/nd:>7.3f}")
    print("\nby recall:", " > ".join(NAME[m] for m in sorted(MET, key=lambda m: -FLOOR[m]["recall"])))
    print("by excess:", " > ".join(NAME[m] for m in sorted(MET, key=lambda m: -FLOOR[m]["excess"])))

method            recall  chance  excess   prec.
SegmentAnyTree     0.676   0.516   0.159   0.840


AMS3D              0.643   0.503   0.140   0.832


ForestFormer3D     0.611   0.478   0.133   0.840
TreeLite3D         0.564   0.507   0.057   0.693


CHM watershed      0.410   0.312   0.098   0.936

by recall: SegmentAnyTree > AMS3D > ForestFormer3D > TreeLite3D > CHM watershed
by excess: SegmentAnyTree > AMS3D > ForestFormer3D > CHM watershed > TreeLite3D


## 5. Results by site and the density correlations

In [8]:
if not FULL:
    print(SKIP)
else:
    def paired_diff(a, b, keys, nb=NBOOT, seed=0):
        va, vb = [TAB[a][k] for k in keys], [TAB[b][k] for k in keys]
        d0 = (sum(x["TP"] for x in va) / sum(x["NR"] for x in va)
              - sum(x["TP"] for x in vb) / sum(x["NR"] for x in vb))
        rng = np.random.default_rng(seed)
        out = np.empty(nb)
        for j in range(nb):
            i = rng.integers(0, len(keys), len(keys))
            na = sum(va[t]["TP"] for t in i); da = sum(va[t]["NR"] for t in i)
            mb = sum(vb[t]["TP"] for t in i); db = sum(vb[t]["NR"] for t in i)
            out[j] = (na / da if da else 0) - (mb / db if db else 0)
        return d0, np.percentile(out, [2.5, 97.5])


    GAP_P = {}
    print(f"{'method':16} {'Chamrousse':>24} {'Pellizzano':>24} {'Both':>24} {'rho':>6} {'p':>7}")
    for m in MET[1:]:
        row = f"{NAME[m]:16}"
        for g in ("Chamrousse", "Pellizzano", "Both"):
            d0, (lo, hi) = paired_diff("SAT", m, GRP[g])
            row += f"  {d0:+.3f} [{lo:+.3f},{hi:+.3f}]"
        gap = [TAB["SAT"][k]["TP"] / TAB["SAT"][k]["NR"] - TAB[m][k]["TP"] / TAB[m][k]["NR"]
               for k in KEYS]
        rho, p = spearman(gap, [DENS[k]["density"] for k in KEYS])
        GAP_P[m] = p
        print(row + f" {rho:>6.2f} {p:>7.3f}")

    REC_P = {}
    for m in MET:
        rec = [TAB[m][k]["TP"] / TAB[m][k]["NR"] for k in KEYS]
        rho, p = spearman(rec, [DENS[k]["density"] for k in KEYS])
        REC_P[m] = p
        print(f"recall vs density  {NAME[m]:16} rho {rho:+.2f}  p {p:.4f}")

method                         Chamrousse               Pellizzano                     Both    rho       p


AMS3D             +0.064 [+0.020,+0.089]  -0.018 [-0.066,+0.021]  +0.033 [-0.014,+0.060]  -0.47   0.028


ForestFormer3D    +0.107 [+0.070,+0.187]  -0.006 [-0.042,+0.025]  +0.065 [+0.027,+0.102]  -0.65   0.001


TreeLite3D        +0.213 [+0.096,+0.278]  -0.053 [-0.125,+0.030]  +0.112 [-0.015,+0.190]  -0.50   0.020


CHM watershed     +0.275 [+0.210,+0.305]  +0.250 [+0.204,+0.299]  +0.266 [+0.221,+0.293]  -0.07   0.744


recall vs density  SegmentAnyTree   rho +0.16  p 0.4797


recall vs density  AMS3D            rho +0.31  p 0.1625


recall vs density  ForestFormer3D   rho +0.44  p 0.0381


recall vs density  TreeLite3D       rho +0.74  p 0.0001


recall vs density  CHM watershed    rho +0.18  p 0.4116


## 6. Multiplicity

In [9]:
if not FULL:
    print(SKIP)
else:
    def benjamini_hochberg(p):
        m = len(p)
        order = np.argsort(p)
        q = np.empty(m)
        prev = 1.0
        for rank, i in enumerate(reversed(order), start=1):
            prev = min(prev, p[i] * m / (m - rank + 1))
            q[i] = prev
        return q


    for fam, tests in (("gap vs density", GAP_P), ("recall vs density", REC_P)):
        ks = list(tests)
        q = benjamini_hochberg([tests[k] for k in ks])
        print(f"\n{fam}, m = {len(ks)}")
        for k, qq in zip(ks, q):
            print(f"   {NAME[k]:16} p {tests[k]:>7.4f}  q {qq:>7.4f}  "
                  f"{'survives' if qq < 0.05 else 'does not'}")


gap vs density, m = 4
   AMS3D            p  0.0284  q  0.0379  survives
   ForestFormer3D   p  0.0014  q  0.0056  survives
   TreeLite3D       p  0.0202  q  0.0379  survives
   CHM watershed    p  0.7438  q  0.7438  does not

recall vs density, m = 5
   SegmentAnyTree   p  0.4797  q  0.4797  does not
   AMS3D            p  0.1625  q  0.2708  does not
   ForestFormer3D   p  0.0381  q  0.0952  does not
   TreeLite3D       p  0.0001  q  0.0005  survives
   CHM watershed    p  0.4116  q  0.4797  does not


## 7. Detectability strata

In [10]:
if not FULL:
    print(SKIP)
else:
    STRATA = ("dominant", "intermediate", "suppressed")
    BARS = {}
    for s in STRATA:
        row = {}
        n = 0
        for m in MET:
            tp = nr = 0
            for k in KEYS:
                st = np.array(REF[k]["stratum"])
                hit = np.zeros(len(st), bool)
                for _, j in TAB[m][k]["pairs"]:
                    hit[j] = True
                tp += int(hit[st == s].sum()); nr += int((st == s).sum())
            row[m] = tp / nr
            n = nr
        BARS[s] = row
        print(f"{s:14} n={n:>4}  " + "  ".join(f"{NAME[m][:6]} {row[m]:.3f}" for m in MET))

dominant       n= 848  Segmen 0.807  AMS3D 0.767  Forest 0.738  TreeLi 0.691  CHM wa 0.571
intermediate   n= 381  Segmen 0.559  AMS3D 0.509  Forest 0.459  TreeLi 0.360  CHM wa 0.186
suppressed     n= 208  Segmen 0.356  AMS3D 0.385  Forest 0.370  TreeLi 0.418  CHM wa 0.163


## 8. Controlled thinning

In [11]:
# pulses removed on GPU-side runners; the stored detections are loaded here
#   python3 modal/thin_clouds.py
#   modal run modal/run_sat.py --files "pell_p1_f60.las,..."
#   modal run modal/run_ff3d.py --names "pell_p1_f60,..."
#   modal run modal/run_treelite.py::main --plots "p1,..."
LAB = {"SAT": "SAT", "FF3D": "FF3D", "t4_p30": "TreeLite3D"}
PS = [f"p{i}" for i in range(1, 16)]
D2 = json.load(open(os.path.join(RESULTS, "thinning_density.json")))
SUB = {m: json.load(open(os.path.join(RESULTS, f"thinning_{f}.json")))
       for m, f in (("SAT", "SAT"), ("FF3D", "FF3D"), ("TreeLite3D", "t4_p30"))}
FR = ["100", "60", "40", "25"]

print(f"{'pulses':>7} {'returns/m2':>11}" + "".join(f"{NAME[m]:>18}" for m in SUB))
for f in FR:
    row = f"{f+' %':>7} {D2[f]:>11.1f}"
    for m, v in SUB.items():
        lo, hi = v[f]["ci"]
        row += f"   {v[f]['recall']:.3f} [{lo:.2f},{hi:.2f}]"
    print(row)

 pulses  returns/m2    SegmentAnyTree    ForestFormer3D        TreeLite3D
  100 %       118.1   0.624 [0.54,0.73]   0.630 [0.56,0.74]   0.678 [0.61,0.75]
   60 %        70.8   0.580 [0.51,0.68]   0.547 [0.49,0.64]   0.600 [0.51,0.69]
   40 %        47.1   0.575 [0.51,0.66]   0.477 [0.42,0.56]   0.473 [0.40,0.57]
   25 %        29.5   0.545 [0.48,0.64]   0.372 [0.32,0.44]   0.210 [0.14,0.31]


In [12]:
if not FULL:
    print(SKIP)
else:
    def thinned_recall(met_key, frac):
        out = {}
        sub = None if frac == 100 else json.load(open(os.path.join(
            ROOT, "treelite_output", f"thinning_detections_{met_key}.json")))
        for p in PS:
            ref, reg = REF[f"pell/{p}"], REG[f"pell/{p}"]
            if frac == 100:
                d = DET[f"pell/{p}"][LAB[met_key]]
            else:
                k = f"pell_{p}_f{frac}"
                if k not in sub:
                    continue
                d = sub[k]
            x, y = np.array(d["x"]), np.array(d["y"])
            ins = reg.contains_points(np.c_[x, y]) if len(x) else np.zeros(0, bool)
            out[p] = dict(TP=len(greedy(x[ins], y[ins], ref["x"], ref["y"])),
                          ND=int(ins.sum()), NR=len(ref["x"]))
        return out


    PAIRED = {}
    print(f"{'model':16} {'mean drop':>10} {'95% CI':>17} {'plots down':>11} {'exact p':>10}")
    for mk in ("SAT", "FF3D", "t4_p30"):
        a, b = thinned_recall(mk, 100), thinned_recall(mk, 25)
        ks = [p for p in PS if p in a and p in b]
        dd = np.array([a[p]["TP"] / a[p]["NR"] - b[p]["TP"] / b[p]["NR"] for p in ks])
        obs = float(dd.mean())
        hits = sum(1 for s in itertools.product([1, -1], repeat=len(dd))
                   if abs(np.dot(s, dd) / len(dd)) >= abs(obs) - 1e-12)
        pv = hits / 2 ** len(dd)
        rng2 = np.random.default_rng(0)
        bs = np.array([dd[rng2.integers(0, len(dd), len(dd))].mean() for _ in range(NBOOT)])
        lo, hi = np.percentile(bs, [2.5, 97.5])
        PAIRED[LAB[mk]] = dict(drop=obs, p=pv, down=int((dd > 0).sum()), n=len(dd))
        print(f"{NAME[LAB[mk]]:16} {obs:>10.3f} [{lo:>6.3f},{hi:>6.3f}] "
              f"{int((dd>0).sum()):>6}/{len(dd):<4} {pv:>10.5f}")

model             mean drop            95% CI  plots down    exact p
SegmentAnyTree        0.109 [ 0.053, 0.179]     13/15      0.00171
ForestFormer3D        0.311 [ 0.234, 0.385]     14/15      0.00012


TreeLite3D            0.424 [ 0.295, 0.537]     14/15      0.00024


In [13]:
if not FULL:
    print(SKIP)
else:
    print(f"{'model':16} {'pulses':>7} {'ND':>6} {'recall':>7} {'chance':>7} {'excess':>7} {'prec.':>7}")
    THIN = {}
    for mk in ("SAT", "FF3D", "t4_p30"):
        THIN[LAB[mk]] = {}
        for fr in (100, 25):
            pp = thinned_recall(mk, fr)
            tp = sum(v["TP"] for v in pp.values())
            nd = sum(v["ND"] for v in pp.values())
            nr = sum(v["NR"] for v in pp.values())
            tp0 = sum(uniform_recall(pp[p]["ND"], REG[f"pell/{p}"], REF[f"pell/{p}"], DMATCH, rng)
                      for p in pp)
            THIN[LAB[mk]][fr] = dict(recall=tp / nr, chance=tp0 / nr,
                                     excess=tp / nr - tp0 / nr, ND=nd, prec=tp / nd)
            r = THIN[LAB[mk]][fr]
            print(f"{NAME[LAB[mk]]:16} {fr:>6}% {nd:>6} {r['recall']:>7.3f} {r['chance']:>7.3f} "
                  f"{r['excess']:>7.3f} {r['prec']:>7.3f}")
    print()
    for k, v in THIN.items():
        print(f"{NAME[k]:16} recall drop {v[100]['recall']-v[25]['recall']:>6.3f}   "
              f"excess drop {v[100]['excess']-v[25]['excess']:>+.3f}   "
              f"detections {v[100]['ND']} to {v[25]['ND']}   "
              f"precision {v[100]['prec']:.3f} to {v[25]['prec']:.3f}")

model             pulses     ND  recall  chance  excess   prec.
SegmentAnyTree      100%    382   0.624   0.494   0.130   0.887
SegmentAnyTree       25%    334   0.545   0.440   0.105   0.886
ForestFormer3D      100%    408   0.630   0.509   0.120   0.838
ForestFormer3D       25%    214   0.372   0.309   0.063   0.944
TreeLite3D          100%    576   0.678   0.645   0.032   0.639
TreeLite3D           25%    131   0.210   0.174   0.035   0.870

SegmentAnyTree   recall drop  0.079   excess drop +0.025   detections 382 to 334   precision 0.887 to 0.886
ForestFormer3D   recall drop  0.258   excess drop +0.058   detections 408 to 214   precision 0.838 to 0.944
TreeLite3D       recall drop  0.468   excess drop -0.003   detections 576 to 131   precision 0.639 to 0.870


## 9. TreeLite3D: defaults, sweep, leave-one-plot-out

In [14]:
TL = json.load(open(os.path.join(RESULTS, "treelite_evaluation.json")))
tested = TL["tunings_tested"]
print(f"settings tested: {len(tested)} = 1 published default ({TL['default']['tuning']})"
      f" + {len(tested) - 1} swept   selected: {TL['best']}")
print(" ", tested)
for k in ("default", "oracle", "leave_one_out"):
    v = TL[k]["Both"]
    print(f"{k:16} recall {v['recall']:.3f}  TP {v['TP']}/{v['NR']}")
sel = TL["leave_one_out"]["selected"]
print("per-plot selection unique values:", sorted(set(sel.values())))

settings tested: 23 = 1 published default (t1) + 22 swept   selected: t4_p30
  ['t1', 't10', 't2', 't3', 't3_p12', 't3_p30', 't4', 't4_p12', 't4_p20', 't4_p30', 't4_p50', 't5', 't5_p12', 't5_p20', 't5_p30', 't5_p50', 't6', 't6_p12', 't6_p20', 't6_p30', 't6_p50', 't8', 't8_p30']
default          recall 0.075  TP 108/1437
oracle           recall 0.564  TP 810/1437
leave_one_out    recall 0.564  TP 810/1437
per-plot selection unique values: ['t4_p30']


## 10. Evaluation frame

In [15]:
if not FULL:
    print(SKIP)
else:
    for dm in (2.5, 3.5, 5.0):
        tab = {m: {k: per_plot(k, m, dm) for k in KEYS} for m in MET}
        line = f"d_match {dm:>4} m  "
        for m in MET:
            TP, ND, NR, r, p, f = pooled(m, KEYS, tab)
            line += f"{NAME[m][:6]} {r:.3f}  "
        print(line)

    RC = json.load(open(os.path.join(RESULTS, "review_checks.json")))
    print("\nPellizzano framings:", list(RC["D_region_pellizzano"]["SAT"]))
    for mode in RC["D_region_pellizzano"]["SAT"]:
        line = f"{mode:26}"
        for m in ("SAT", "AMS3D", "WS"):
            v = RC["D_region_pellizzano"][m][mode]
            line += f"  {NAME[m][:6]} r {v['recall']:.3f} p {v['precision']:.3f}"
        print(line)

d_match  2.5 m  Segmen 0.635  AMS3D 0.607  Forest 0.565  TreeLi 0.502  CHM wa 0.399  
d_match  3.5 m  Segmen 0.676  AMS3D 0.643  Forest 0.611  TreeLi 0.564  CHM wa 0.410  
d_match  5.0 m  Segmen 0.709  AMS3D 0.682  Forest 0.653  TreeLi 0.619  CHM wa 0.420  

Pellizzano framings: ['hyperspectral footprint', 'hull dilated 1.5 m', 'whole tile']
hyperspectral footprint     Segmen r 0.676 p 0.470  AMS3D r 0.678 p 0.771  CHM wa r 0.418 p 0.510
hull dilated 1.5 m          Segmen r 0.624 p 0.887  AMS3D r 0.643 p 0.831  CHM wa r 0.374 p 0.990
whole tile                  Segmen r 0.676 p 0.441  AMS3D r 0.678 p 0.771  CHM wa r 0.418 p 0.483


## 11. Crown Jaccard and species

In [16]:
J = json.load(open(os.path.join(RESULTS, "crown_jaccard.json")))
for m in MET:
    print(f"{NAME[m]:16} J " +
          "  ".join(f"{g} {J[g][m]['J']:.3f}" for g in ("Chamrousse", "Pellizzano")))

SP = json.load(open(os.path.join(RESULTS, "species_recall.json")))
print()
for sp, v in SP.items():
    if not (isinstance(v, dict) and "n" in v):
        continue
    cells = "  ".join(f"{NAME[m][:6]} {v[m]['recall']:.3f}"
                      for m in MET if isinstance(v.get(m), dict) and "recall" in v[m])
    print(f"{sp:24} n {v['n']:>4}  {cells}")

SegmentAnyTree   J Chamrousse 0.529  Pellizzano 0.482
AMS3D            J Chamrousse 0.463  Pellizzano 0.468
ForestFormer3D   J Chamrousse 0.511  Pellizzano 0.458
TreeLite3D       J Chamrousse 0.372  Pellizzano 0.340
CHM watershed    J Chamrousse 0.542  Pellizzano 0.531

Abies alba               n  295  Segmen 0.654  AMS3D 0.610  Forest 0.620  TreeLi 0.441  CHM wa 0.373
Betula pendula           n   31  Segmen 0.484  AMS3D 0.710  Forest 0.484  TreeLi 0.452  CHM wa 0.290
Fagus sylvatica          n  138  Segmen 0.529  AMS3D 0.471  Forest 0.399  TreeLi 0.370  CHM wa 0.275
Larix decidua            n   53  Segmen 0.868  AMS3D 0.774  Forest 0.887  TreeLi 0.755  CHM wa 0.698
Picea abies              n  746  Segmen 0.713  AMS3D 0.670  Forest 0.654  TreeLi 0.670  CHM wa 0.446
Pinus uncinata           n   95  Segmen 0.737  AMS3D 0.674  Forest 0.589  TreeLi 0.337  CHM wa 0.432
All conifers             n 1137  Segmen 0.700  AMS3D 0.655  Forest 0.640  TreeLi 0.583  CHM wa 0.427
All broadleaves       

## 12. Overview figure

In [17]:
if not FULL:
    print(SKIP)
else:
    fig, a = plt.subplots(figsize=(2.86, 1.86))
    xs = [D2[f] for f in FR]
    cd = [DENS[k]["density"] for k in KEYS if k.startswith("cham")]
    a.axvspan(min(cd), max(cd), color="0.92", zorder=0, lw=0)
    a.text((min(cd) + max(cd)) / 2, 0.055, "Chamrousse\nrange", fontsize=6.2, ha="center",
           color="0.45", va="bottom")
    for j, m in enumerate(("SAT", "FF3D", "TreeLite3D")):
        v = SUB[m]
        y = [v[f]["recall"] for f in FR]
        a.fill_between(xs, [v[f]["ci"][0] for f in FR], [v[f]["ci"][1] for f in FR],
                       color=COL[m], alpha=.13, lw=0, zorder=1)
        a.plot(xs, y, marker=MRK[m], ms=4.2, color=COL[m], lw=1.9, zorder=3)
        a.text(.985, .045 + .088 * (2 - j), NAME[m], transform=a.transAxes,
               fontsize=6.6, color=COL[m], ha="right", va="bottom")
    a.set_xlabel("Return density (returns/m$^2$)", fontsize=7.5)
    a.set_ylabel("Plot recall", fontsize=7.5)
    a.set_xlim(20, 128); a.set_ylim(0, .82)
    a.grid(lw=.3, color="0.9"); a.set_axisbelow(True); a.tick_params(labelsize=6.8)
    fig.tight_layout(pad=0.15)
    fig.savefig("nb_fig1a.png", dpi=180)
    print("panel (a)")

panel (a)


In [18]:
if not FULL:
    print(SKIP)
else:
    fig, c = plt.subplots(figsize=(3.94, 1.86))
    NAMES = [f"Dominant\n$n$ = {sum(1 for k in KEYS for s in REF[k]['stratum'] if s=='dominant')}",
           f"Intermediate\n$n$ = {sum(1 for k in KEYS for s in REF[k]['stratum'] if s=='intermediate')}",
           f"Suppressed\n$n$ = {sum(1 for k in KEYS for s in REF[k]['stratum'] if s=='suppressed')}"]
    w, x0 = .165, np.arange(3)
    for j, m in enumerate(MET):
        c.bar(x0 + (j - 2) * w, [BARS[s][m] for s in STRATA], w * .88, color=COL[m],
              edgecolor="white", linewidth=.4, label=NAME[m], zorder=2)
    c.set_xticks(x0); c.set_xticklabels(NAMES, fontsize=7)
    c.set_ylabel("Recall", fontsize=7.5); c.set_ylim(0, .98)
    c.legend(fontsize=6.2, handlelength=1.0, labelspacing=.2, loc="upper right", ncol=2,
             columnspacing=.8)
    c.grid(axis="y", lw=.3, color="0.9"); c.set_axisbelow(True); c.tick_params(labelsize=6.8)
    fig.tight_layout(pad=0.15)
    fig.savefig("nb_fig1b.png", dpi=180)
    print("panel (b)")

panel (b)


## 13. Single trees

In [19]:
if not FULL:
    print(SKIP)
else:
    import laspy
    from matplotlib.path import Path

    CROWNS = json.load(open(os.path.join(DATA, "viewer", "data", "crowns.json"))) \
        if os.path.exists(os.path.join(DATA, "viewer", "data", "crowns.json")) else None
    TREES = (("cham", "4", 86), ("pell", "p8", 54))
    GREY = "#c2c6cc"


    def cloud_sat(site, p):
        L = laspy.read(os.path.join(ROOT, "sat_output_all", f"{site}_{p}_out.laz"))
        return (np.c_[np.asarray(L.x), np.asarray(L.y), np.asarray(L.z)],
                np.asarray(L.PredInstance).astype(np.int64))


    def cloud_tl(site, p):
        r = (os.path.join(DATA, "IRSTEA_dataset", "las", "full_las", f"las{p}.las")
             if site == "cham" else os.path.join(DATA, "trento_dataset", "las", f"{p}.las"))
        L = laspy.read(r)
        e = np.load(os.path.join(ROOT, "treelite_output", "t4_p30", f"instance_pred_{site}_{p}.npy")).reshape(-1)
        m = np.load(os.path.join(ROOT, "treelite_output", "t4_p30", f"assigned_{site}_{p}.npy")).reshape(-1)
        return np.c_[np.asarray(L.x), np.asarray(L.y), np.asarray(L.z)], np.where(m, e + 1, 0)


    cm20 = plt.get_cmap("tab20")
    fig, axes = plt.subplots(2, 3, figsize=(6.4, 4.0),
                             subplot_kw={"projection": "3d"})
    for row, (site, p, i) in enumerate(TREES):
        key = f"{site}/{p}"
        inv = REF[key]
        polys = CROWNS[key]["inv"] if CROWNS else []
        if i < len(polys):
            pol = np.asarray(polys[i], float) / 10.0
        else:
            rad = float(np.sqrt(inv["area"][i] / np.pi))
            th = np.linspace(0, 2 * np.pi, 33)[:-1]
            pol = np.c_[rad * np.cos(th), rad * np.sin(th)]
        x0, y0, h = inv["x"][i], inv["y"][i], inv["h"][i]
        rmax = np.hypot(pol[:, 0], pol[:, 1]).max()
        path = Path(np.c_[pol[:, 0] + x0, pol[:, 1] + y0])
        xs_, is_ = cloud_sat(site, p)
        xt_, it_ = cloud_tl(site, p)
        for col, (pts, inst) in enumerate(((xs_, None), (xs_, is_), (xt_, it_))):
            d = np.hypot(pts[:, 0] - x0, pts[:, 1] - y0)
            m = (d <= rmax * 1.25) & (pts[:, 2] >= 1.0) & (pts[:, 2] <= h + 2)
            q = pts[m]
            inside = path.contains_points(q[:, :2]) & (q[:, 2] >= max(1.5, .4 * h))
            if inst is None:
                colr = np.tile(matplotlib.colors.to_rgb("#8a9099"), (len(q), 1))
            else:
                v = inst[m]
                colr = np.where((v > 0)[:, None], cm20((v % 20) / 20.0)[:, :3],
                                np.array(matplotlib.colors.to_rgb(GREY))[None, :])
            ax = axes[row, col]
            o = np.argsort(q[:, 2])
            dd = inside[o]
            ax.scatter(q[o][~dd, 0], q[o][~dd, 1], q[o][~dd, 2], c=colr[o][~dd], s=.9,
                       lw=0, alpha=.3, depthshade=False)
            ax.scatter(q[o][dd, 0], q[o][dd, 1], q[o][dd, 2], c=colr[o][dd], s=2.4, lw=0,
                       depthshade=False)
            px, py = pol[:, 0] + x0, pol[:, 1] + y0
            for z in (max(1.5, .4 * h), h):
                ax.plot(np.r_[px, px[0]], np.r_[py, py[0]], z, color="0.25", lw=.55)
            ax.set_axis_off(); ax.view_init(elev=14, azim=-58)
            ax.set_xlim(x0 - rmax * 1.3, x0 + rmax); ax.set_ylim(y0 - rmax, y0 + rmax)
            ax.set_zlim(0, h + 2)
            ax.set_box_aspect((1, 1, min(2.6, (h + 2) / (2 * rmax))), zoom=1.45 if (h + 2) / (2 * rmax) < 1 else 1.05)
            n_in = int(inside.sum())
            n_as = int((inst[m][inside] > 0).sum()) if inst is not None else 0
            ax.set_title(["field reference", "SegmentAnyTree", "TreeLite3D"][col] +
                         ("" if inst is None else f"\n{n_as} of {n_in} returns"), fontsize=7)
    fig.tight_layout()
    fig.savefig("nb_fig2.png", dpi=180)
    print("figure 2")

figure 2


## 14. Against the reported values

In [20]:
if not FULL:
    print(SKIP)
else:
    PUB = {("Both", "SAT"): 0.676, ("Both", "AMS3D"): 0.643, ("Both", "FF3D"): 0.611,
           ("Both", "TreeLite3D"): 0.564, ("Both", "WS"): 0.410,
           ("Chamrousse", "SAT"): 0.707, ("Pellizzano", "SAT"): 0.624}
    ok = True
    for (g, m), want in PUB.items():
        got = pooled(m, GRP[g])[3]
        good = abs(got - want) < 5e-4
        ok &= good
        print(f"{g:11} {NAME[m]:16} {got:.3f} vs {want:.3f}  {'ok' if good else 'MISMATCH'}")

    CF = json.load(open(os.path.join(RESULTS, "chance_floor.json")))["by_tolerance"]["3.5"]
    for m in MET:
        good = abs(FLOOR[m]["excess"] - CF[m]["excess"]) < 0.02
        ok &= good
        print(f"excess      {NAME[m]:16} {FLOOR[m]['excess']:.3f} vs {CF[m]['excess']:.3f}  "
              f"{'ok' if good else 'MISMATCH'}")
    print("\nall match" if ok else "\nsomething moved")

Both        SegmentAnyTree   0.676 vs 0.676  ok
Both        AMS3D            0.643 vs 0.643  ok
Both        ForestFormer3D   0.611 vs 0.611  ok
Both        TreeLite3D       0.564 vs 0.564  ok
Both        CHM watershed    0.410 vs 0.410  ok
Chamrousse  SegmentAnyTree   0.707 vs 0.707  ok
Pellizzano  SegmentAnyTree   0.624 vs 0.624  ok
excess      SegmentAnyTree   0.159 vs 0.161  ok
excess      AMS3D            0.140 vs 0.140  ok
excess      ForestFormer3D   0.133 vs 0.135  ok
excess      TreeLite3D       0.057 vs 0.055  ok
excess      CHM watershed    0.098 vs 0.097  ok

all match
